<a href="https://colab.research.google.com/github/harshitadalne/AI-Practicals-Python-/blob/main/NNDL_Practical_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

VOCAB_SIZE = 10000
MAX_LENGTH = 200
EMBEDDING_DIM = 128

print("Loading IMDB dataset...")

(X_train, y_train), (X_test, y_test) = imdb.load_data(
    num_words=VOCAB_SIZE
)

print("\nDataset loaded successfully!")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nPadding sequences...")

X_train = pad_sequences(
    X_train,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

X_test = pad_sequences(
    X_test,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

model = Sequential([
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM
    ),
    LSTM(
        64,
        dropout=0.2,
        recurrent_dropout=0.2
    ),
    Dropout(0.3),
    Dense(
        1,
        activation="sigmoid"
    )
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nModel Summary:")
model.summary()

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

print("\nTraining model...")

history = model.fit(
    X_train,
    y_train,
    epochs=8,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\nTest Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_accuracy * 100, 2), "%")

word_index = imdb.get_word_index()

def encode_text(text):
    words = text.lower().split()
    encoded = []

    for word in words:
        word = word.strip(".,!?;:\"'()[]{}")

        if word in word_index:
            index = word_index[word] + 3

            if index < VOCAB_SIZE:
                encoded.append(index)
            else:
                encoded.append(2)
        else:
            encoded.append(2)

    return encoded


def predict_sentiment(text):
    encoded_text = encode_text(text)

    padded_text = pad_sequences(
        [encoded_text],
        maxlen=MAX_LENGTH,
        padding="post",
        truncating="post"
    )

    prediction = model.predict(
        padded_text,
        verbose=0
    )[0][0]

    if prediction >= 0.5:
        sentiment = "Positive"
        confidence = prediction * 100
    else:
        sentiment = "Negative"
        confidence = (1 - prediction) * 100

    return sentiment, confidence


test_sentences = [
    "I absolutely loved this movie. It was fantastic and entertaining.",
    "This movie was amazing and the acting was excellent.",
    "I really enjoyed this film and would recommend it to everyone.",
    "The story was beautiful and the characters were wonderful.",
    "This was one of the best movies I have ever watched.",
    "The movie was terrible and extremely boring.",
    "I hated this film because the acting was awful.",
    "This was a complete waste of time and money.",
    "The story was boring and the characters were disappointing.",
    "I would never recommend this horrible movie."
]

print("\nSentiment Predictions:")

for sentence in test_sentences:
    sentiment, confidence = predict_sentiment(sentence)

    print("\nSentence:", sentence)
    print("Sentiment:", sentiment)
    print("Confidence:", round(confidence, 2), "%")


print("\nCustom Sentiment Analysis")
print("Type 'exit' to stop the program.")

while True:
    user_text = input("\nEnter a sentence: ")

    if user_text.lower() == "exit":
        print("Program ended.")
        break

    if user_text.strip() == "":
        print("Please enter a valid sentence.")
        continue

    sentiment, confidence = predict_sentiment(user_text)

    print("Predicted Sentiment:", sentiment)
    print("Confidence:", round(confidence, 2), "%")

Loading IMDB dataset...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Dataset loaded successfully!
Training samples: 25000
Testing samples: 25000

Padding sequences...
Training data shape: (25000, 200)
Testing data shape: (25000, 200)

Model Summary:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training model...
Epoch 1/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 81s 488ms/step - accuracy: 0.5238 - loss: 0.6918 - val_accuracy: 0.5492 - val_loss: 0.6869
Epoch 2/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 80s 474ms/step - accuracy: 0.5951 - loss: 0.6642 - val_accuracy: 0.6758 - val_loss: 0.6405
Epoch 3/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 81s 469ms/step - accuracy: 0.6395 - loss: 0.6432 - val_accuracy: 0.5406 - val_loss: 0.6785
Epoch 4/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 73s 467ms/step - accuracy: 0.6311 - loss: 0.6179 - val_accuracy: 0.6452 - val_loss: 0.6260
Epoch 5/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 75s 478ms/step - accuracy: 0.7045 - loss: 0.5629 - val_accuracy: 0.7690 - val_loss: 0.5576
Epoch 6/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 85s 500ms/step - accuracy: 0.7929 - loss: 0.4827 - val_accuracy: 0.8080 - val_loss: 0.4817
Epoch 7/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 75s 477ms/step - accuracy: 0.8265 - loss: 0.4367 - val_accuracy: 0.8198 - val_loss: 0.4588
Epoch 8/8
157/157 ━━━━━━━━━━━━━━━━━━━━ 73s 462ms/step - accuracy: 0.844